## Setup and Imports



In [3]:
# %matplotlib inline
!pip install timm

## Download and Organize

In [4]:
import os
import shutil
import zipfile
import urllib.request
import sys

# Colab-specific paths
BASE_DIR = "/content"
DATA_DIR = os.path.join(BASE_DIR, "data", "tomato")
TEMP_DIR = os.path.join(BASE_DIR, "data", "_temp")
HF_ZIP_URL = "https://huggingface.co/datasets/mohanty/PlantVillage/resolve/main/data.zip"

def download_progress(count, block_size, total_size):
    downloaded = count * block_size
    if total_size > 0:
        percent = int(downloaded * 100 / total_size)
        sys.stdout.write(f"\rDownloading... {min(100, percent)}%")
        sys.stdout.flush()

# Download and Extract
os.makedirs(TEMP_DIR, exist_ok=True)
zip_path = os.path.join(TEMP_DIR, "data.zip")

if not os.path.exists(DATA_DIR):
    print("Downloading data...")
    urllib.request.urlretrieve(HF_ZIP_URL, zip_path, reporthook=download_progress)

    print("\nExtracting...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(os.path.join(TEMP_DIR, "extracted"))

    # Simple logic to find the tomato folder inside the extracted zip
    # Note: Structure varies, usually it's inside 'color'
    extracted_root = os.path.join(TEMP_DIR, "extracted")

    # Move files to DATA_DIR
    # This logic assumes standard PlantVillage structure
    for root, dirs, files in os.walk(extracted_root):
        if "Tomato___Bacterial_spot" in dirs: # Found the class folders
            shutil.copytree(root, DATA_DIR)
            break
    print("Data ready at:", DATA_DIR)
else:
    print("Data already exists at:", DATA_DIR)

Downloading... 100%
Extracting...
Data ready at: /content/data/tomato


# Imports and Configuration

In [5]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import timm
import matplotlib.pyplot as plt
import random

# Configuration
config = {
    "data_dir": "/content/data/tomato",
    "epochs": 3,
    "batch_size": 32,
    "lr": 1e-3,
    "output_dir": "/content/checkpoints"
}

# Ensure output dir exists
os.makedirs(config["output_dir"], exist_ok=True)

# Set seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Data Loaders and Model Setup

In [6]:
# Reuse your original transforms logic
t_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

v_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create Dataset
full_ds = datasets.ImageFolder(config["data_dir"], transform=t_transforms)
indices = list(range(len(full_ds)))
random.shuffle(indices)
split_pos = int(len(full_ds) * 0.8)

train_loader = DataLoader(Subset(full_ds, indices[:split_pos]), batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(Subset(datasets.ImageFolder(config["data_dir"], transform=v_transforms), indices[split_pos:]), batch_size=config["batch_size"])

# Model
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(full_ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

## Training Loop

In [7]:
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, config["epochs"] + 1):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Simple validation log
    avg_train_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch} finished. Training Loss: {avg_train_loss:.4f}")

# Save the final model
torch.save(model.state_dict(), os.path.join(config["output_dir"], "model.pth"))
print("Model saved to:", config["output_dir"])

Epoch 1 finished. Training Loss: 0.3129
Epoch 2 finished. Training Loss: 0.1436
Epoch 3 finished. Training Loss: 0.1209
Model saved to: /content/checkpoints
